# 🧮 24-信号处理 · 06 z 变换与系统分析

> 关键词：z 变换、传递函数、极点零点、稳定
> 卷积在时域是"翻褶相乘求和"，在 z 域只是一次乘法——系统分析从"算"变成"看图"。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | z 变换定义与收敛域（ROC）？ | §1 核心概念 |
| 2 | 常见序列的 z 变换怎么手算？ | §2 手算 ✍️ |
| 3 | 极点在单位圆内/上/外各意味着什么？ | §3 深入原理 |
| 4 | 频率响应怎么从零极点图读出来？ | §4 代码实战 |

## 故事引入

两个 LTI 系统串联，时域要算一遍长卷积；在 z 域只需把两个传递函数乘起来。更重要的是：**把 H(z) 的极点点在复平面上，系统的稳定与频响一眼可读**——这是信号与系统最优雅的一张图。

## 核心概念

**z 变换**：

$$X(z)=\sum_{n=-\infty}^{\infty}x[n]z^{-n}$$

**常见变换对**：

| 序列 | z 变换 | ROC |
| --- | --- | --- |
| $\delta[n]$ | 1 | 全平面 |
| $u[n]$ | $z/(z-1)$ | $|z|>1$ |
| $a^n u[n]$ | $1/(1-az^{-1})$ | $|z|>|a|$ |
| $\delta[n-k]$ | $z^{-k}$ | $z\ne0$ |

**核心性质**：卷积定理 $x*h \leftrightarrow X(z)H(z)$；时移 $x[n-k]\leftrightarrow z^{-k}X(z)$。

## ✍️ 手算 z 变换与频响

**设定 A：有限序列的 z 变换**

$$h=[1,\ 0.5,\ 0.25]\Rightarrow H(z)=1+0.5z^{-1}+0.25z^{-2}$$

在 $z=2$ 处：$H(2)=1+0.25+0.0625=\mathbf{1.3125}$。

**设定 B：一阶 IIR 的频率响应（闭环 §04）**

$$H(z)=\frac{0.5}{1-0.5z^{-1}},\quad z=e^{j\omega}$$

$$|H(0)|=\frac{0.5}{1-0.5}=\mathbf{1},\qquad |H(\pi)|=\frac{0.5}{1+0.5}=\mathbf{0.333}$$

与 §04 手算的 DC=1、Nyquist=1/3 完全一致——**频响就是 z 域在单位圆上的取值**。

**设定 C：阶跃响应的闭式**

阶跃输入 $X(z)=z/(z-1)$，输出

$$Y(z)=H(z)X(z)=\frac{0.5z}{(z-1)(z-0.5)}\xrightarrow{\text{部分分式}} y[n]=1-(0.5)^{n+1}$$

验证：$n=0,1,2$ → $0.5,0.75,0.875$（§04 逐项一致）。

**手算收获**：

- z 变换把**卷积变乘法**、差分方程变代数方程——复杂度从"算"降为"代"
- 极点是传递函数的"记忆"：$z=0.5$ 的极点对应时域 $0.5^n$——**极点位置直接读出冲激响应形态**
- 部分分式展开是 z 域求逆的核心技能：一阶极点 → 指数序列
- 频响 $H(e^{j\omega})$ = 零极点向量的模商——下面零极点图就是"看图说话

In [1]:
# ---- §2.1 验证：numpy 复算 z 变换 ----
import numpy as np
# 有限序列 z 变换
h = np.array([1.0, 0.5, 0.25])
Hz2 = 1 + 0.5/2 + 0.25/4
print('H(z)=1+0.5z⁻¹+0.25z⁻², z=2 →', Hz2)
assert Hz2 == 1.3125

# 一阶 IIR 频响
def Hiir(w): return 0.5/np.abs(1 - 0.5*np.exp(-1j*w))
print('|H(0)| = %.4f, |H(π)| = %.4f' % (Hiir(0), Hiir(np.pi)))
assert abs(Hiir(0) - 1.0) < 1e-9 and abs(Hiir(np.pi) - 1/3) < 1e-9

# 阶跃响应闭式 y[n]=1-(0.5)^(n+1)
n = np.arange(4)
print('阶跃响应闭式:', np.round(1 - 0.5**(n+1), 4).tolist())
assert np.allclose(1 - 0.5**(n+1), [0.5, 0.75, 0.875, 0.9375])
print('\n✓ 与手算一致')


H(z)=1+0.5z⁻¹+0.25z⁻², z=2 → 1.3125
|H(0)| = 1.0000, |H(π)| = 0.3333
阶跃响应闭式: [0.5, 0.75, 0.875, 0.9375]

✓ 与手算一致


z 变换把差分方程变成传递函数 H(z)=B(z)/A(z)，而分母多项式 A(z) 的根——极点——直接决定系统稳定性。为了看清极点从哪来，先看真实 IIR 滤波器的实现结构：反馈路径上的系数 a_k 正是分母多项式的系数，反馈回路越深，对应极点离原点越远。

![z 变换把差分方程变成传递函数 H(z)=B(z)/A(z)](../figures/iir_structure.jpg) width="85%"

> **怎么读**：图中的反馈回环（下方横向路径）对应 A(z) 各项；极点在 z 平面单位圆内的条件，在结构上就是反馈系数取值不至于让回路"发散"。零极点图、稳定判据与这张结构图三者互通。
>
> 📚 **图源**：Zhu F et al. (2024) *MEAs-Filter*（同上）图 2。


## 深入原理：零极点图与稳定判据

**传递函数**：$H(z)=\frac{B(z)}{A(z)}=\frac{b_0\prod(z-z_i)}{\prod(z-p_i)}$

- **零点** $z_i$：$H(z_i)=0$，陷波（该频率被抑制）
- **极点** $p_i$：$H(p_i)\to\infty$，共振峰（该频率被放大）

**稳定判据**：因果 LTI 稳定 ⇔ 全部极点 $|p_i|<1$（单位圆内）

- $|p|=0.5$：指数衰减 → 稳定
- $|p|=1$：等幅振荡（临界）
- $|p|>1$：指数发散 → 不稳定

**频率响应几何法**：

$$|H(e^{j\omega})|=\frac{\prod|e^{j\omega}-z_i|}{\prod|e^{j\omega}-p_i|}$$

单位圆上的点到零点/极点的距离之比——**极点在圆内靠近单位圆 = 尖峰**（谐振）。

## 代码实战：零极点 → 频响

一阶 IIR 极点 $p=0.5$：从单位圆上走一圈，$\omega=0$ 处距极点最近（$|1-0.5|=0.5$）增益最大 1，$\omega=\pi$ 处最远（$|{-1}-0.5|=1.5$）增益最小 1/3。

In [2]:
# ---- 零极点几何解释频响 ----
import numpy as np
p = 0.5
w = np.array([0.0, np.pi/2, np.pi])
g = 0.5 / np.abs(1 - p*np.exp(-1j*w))
print('ω=0     : 距极点 |1-p| =', abs(1-p), '→ 增益', round(g[0], 3))
print('ω=π/2   : 增益', round(g[1], 3))
print('ω=π     : 距极点 |−1−p| =', abs(-1-p), '→ 增益', round(g[2], 3))
assert abs(g[0] - 1.0) < 1e-9 and abs(g[2] - 1/3) < 1e-9
print('✓ 极点在 ω=0 一侧 → 低通（极点越靠近单位圆，峰越尖）')


ω=0     : 距极点 |1-p| = 0.5 → 增益 1.0
ω=π/2   : 增益 0.447
ω=π     : 距极点 |−1−p| = 1.5 → 增益 0.333
✓ 极点在 ω=0 一侧 → 低通（极点越靠近单位圆，峰越尖）


## 面试考点

**Q1：为什么说 z 变换是信号处理的总纲？**

采样、卷积、差分方程、频率响应、稳定判据都统一在 z 域：$z=e^{j\omega}$ 退化为 DTFT，$z=e^{sT}$ 联系 s 域。

**Q2：极点在单位圆上会怎样？**

冲激响应等幅振荡（如 $z=\pm1$ 的累加器/振荡器），工程上谐振器就把极点放在圆内贴近单位圆。

**Q3：最小相位系统是什么？**

零点和极点都在单位圆内——逆系统稳定，相位延迟最小。

**Q4：和 IIR 稳定性的关系？**

IIR 的反馈系数就是 $A(z)$ 的系数：设计完必须检查 $A(z)$ 的根都在单位圆内（§07 数字化会踩这个坑）。

## 小结与下一章预告

- z 变换：卷积→乘法、差分→代数；$H(z)$ 在 $z=e^{j\omega}$ 就是频响
- 极点决定稳定性：$|p|<1$ 稳定；频响 = 单位圆到零极点的距离比
- 下一章：用这些工具**设计**滤波器——Butterworth/双线性变换/FIR 窗函数法

## ✅ 自测清单

- [ ] 能手算常见序列的 z 变换（δ、u[n]、aⁿu[n]）
- [ ] 能手算一阶 IIR 频响（DC/Nyquist）并联系 §04
- [ ] 能解释零极点图与稳定判据
- [ ] 知道卷积定理与部分分式求逆的思路